In [4]:
import numpy as np

In [5]:
np.random.seed(42)

In [6]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype = float)

y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype = float)

In [7]:
X.shape

(4, 2)

In [8]:
y.shape

(4, 1)

In [9]:
n_input = 2
n_hidden = 4
n_output = 1

In [10]:
np.random.randn(n_input, n_hidden)

array([[ 0.49671415, -0.1382643 ,  0.64768854,  1.52302986],
       [-0.23415337, -0.23413696,  1.57921282,  0.76743473]])

In [11]:
np.random.randn(n_input, n_hidden) * 0.5

array([[-0.23473719,  0.27128002, -0.23170885, -0.23286488],
       [ 0.12098114, -0.95664012, -0.86245892, -0.28114376]])

In [12]:
W1 = np.random.randn(n_input, n_hidden) * 0.5

In [13]:
W1

array([[-0.50641556,  0.15712367, -0.45401204, -0.70615185],
       [ 0.73282438, -0.11288815,  0.0337641 , -0.71237409]])

In [14]:
np.zeros((1, n_hidden))

array([[0., 0., 0., 0.]])

In [15]:
b1 = np.zeros((1, n_hidden))

In [16]:
W2 = np.random.randn(n_hidden, n_output) * 0.5

In [17]:
W2

array([[-0.27219136],
       [ 0.05546129],
       [-0.57549679],
       [ 0.18784901]])

In [18]:
b2 = np.zeros((1, n_output))

In [19]:
b2

array([[0.]])

In [20]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    return a * (1-a)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(int)

In [21]:
W1

array([[-0.50641556,  0.15712367, -0.45401204, -0.70615185],
       [ 0.73282438, -0.11288815,  0.0337641 , -0.71237409]])

In [22]:
X

array([[0., 0.],
       [0., 1.],
       [1., 0.],
       [1., 1.]])

In [23]:
b1

array([[0., 0., 0., 0.]])

In [24]:
def forward(X):
    Z1 = X @ W1 + b1

    A1 = relu(Z1)

    Z2 = A1 @ W2 + b2

    A2 = sigmoid(Z2)

    cache = (Z1, A1, Z2, A2)

    return A2, cache

In [25]:
def compute_loss(A2, y):
    m = y.shape[0]
    loss = -np.mean(y * np.log(A2 + 1e-8) + (1-y) * np.log(1-A2 + 1e-8))

    return loss


In [26]:
m = y.shape[0]

In [27]:
def backward(X, y, cache):
    Z1, A1, Z2, A2 = cache
    m = X.shape[0]

    dL_dZ2 = A2 - y 
    dL_dW2 = (A1.T @ dL_dZ2) / m

    dL_db2 = np.sum(dL_dZ2, axis = 0, keepdims = True) / m

    dL_dA1 = dL_dZ2 @ W2.T  
    dL_dZ1 = dL_dA1 * relu_derivative(Z1) 
    dL_dW1 = (X.T @ dL_dZ1) / m  
    dL_db1 = np.sum(dL_dZ1, axis=0, keepdims=True) / m
    
    return dL_dW1, dL_db1, dL_dW2, dL_db2

In [28]:
lr = 0.5
epochs = 10000

for epoch in range(epochs):
    A2, cache = forward(X)
    loss = compute_loss(A2, y)
    dL_dW1, dL_db1, dL_dW2, dL_db2 = backward(X, y, cache)

    W1 -= lr * dL_dW1
    b1 -= lr * dL_db1
    W2 -= lr * dL_dW2
    b2 -= lr * dL_db2

    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 0.7136
Epoch 1000, Loss: 0.0035
Epoch 2000, Loss: 0.0016
Epoch 3000, Loss: 0.0010
Epoch 4000, Loss: 0.0007
Epoch 5000, Loss: 0.0006
Epoch 6000, Loss: 0.0005
Epoch 7000, Loss: 0.0004
Epoch 8000, Loss: 0.0004
Epoch 9000, Loss: 0.0003


In [29]:
import torch
import torch.nn as nn

In [30]:
X = torch.tensor([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=torch.float32)

y = torch.tensor([
    [0],
    [1],
    [1],
    [0]
], dtype=torch.float32)

In [32]:
class XORNet(nn.Module):
    def __init__(self, n_input=2, n_hidden=4, n_output=1):
        super().__init__()

        self.layer1 = nn.Linear(n_input, n_hidden)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(n_hidden, n_output)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        z1 = self.layer1(x)
        a1 = self.relu(z1)
        z2 = self.layer2(a1)
        a2 = self.sigmoid(z2)
        return a2

model = XORNet(n_input=2, n_hidden=4, n_output=1)

criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 10000
for epoch in range(epochs):
    A2 = model(X)
    loss = criterion(A2, y)

    optimizer.zero_grad()
    loss.backward()
    print("dL/dW1 (layer1.weight.grad):")
    print(model.layer1.weight.grad)   

    print("dL/dW2 (layer2.weight.grad):")
    print(model.layer2.weight.grad)
    optimizer.step()

    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

with torch.no_grad():
    predictions = model(X)
    print("\nPredictions:")
    print(predictions)
    print("\nRounded:")
    print(torch.round(predictions))

dL/dW1 (layer1.weight.grad):
tensor([[-0.0503, -0.0517],
        [ 0.0000,  0.0000],
        [ 0.0082, -0.0004],
        [ 0.0000,  0.0000]])
dL/dW2 (layer2.weight.grad):
tensor([[ 0.0088,  0.0000, -0.0256,  0.0000]])
Epoch 0, Loss: 0.6940
dL/dW1 (layer1.weight.grad):
tensor([[-0.0501, -0.0515],
        [ 0.0000,  0.0000],
        [ 0.0085, -0.0003],
        [ 0.0000,  0.0000]])
dL/dW2 (layer2.weight.grad):
tensor([[ 0.0071,  0.0000, -0.0256,  0.0000]])
dL/dW1 (layer1.weight.grad):
tensor([[-0.0500, -0.0513],
        [ 0.0000,  0.0000],
        [ 0.0088, -0.0003],
        [ 0.0000,  0.0000]])
dL/dW2 (layer2.weight.grad):
tensor([[ 0.0055,  0.0000, -0.0256,  0.0000]])
dL/dW1 (layer1.weight.grad):
tensor([[-0.0004, -0.0017],
        [ 0.0000,  0.0000],
        [ 0.0092, -0.0003],
        [ 0.0000,  0.0000]])
dL/dW2 (layer2.weight.grad):
tensor([[ 0.0047,  0.0000, -0.0256,  0.0000]])
dL/dW1 (layer1.weight.grad):
tensor([[-0.0005, -0.0017],
        [ 0.0000,  0.0000],
        [ 0.0095, -0.